In [0]:
import sys
from pyspark.sql.functions import col, concat_ws, current_timestamp, max, expr
from pyspark.sql.types import *
from pyspark.sql.utils import AnalysisException

parent_path = '/Workspace/Users/esymphony.life@gmail.com/MSF_test/customer'
if parent_path not in sys.path:
    sys.path.insert(0, parent_path)

from common.functions import recursive_explode

In [0]:
def infer_datatypes(df):
    """
    Infers types using try_cast. 
    """
    priority_types = [
        IntegerType(),
        LongType(),
        DoubleType(),
        DecimalType(38, 18),
        DateType(),
        TimestampType(),
        BooleanType()
    ]

    sample_df = df.limit(10000)
    best_types = {}

    for field in df.schema.fields:
        col_name = field.name
        if not isinstance(field.dataType, StringType):
            best_types[col_name] = field.dataType
            continue

        found_type = StringType()
        for t in priority_types:
            type_sql = t.simpleString()

            test = sample_df.select(
                col(col_name),
                expr(f"try_cast(`{col_name}` AS {type_sql})").alias("cast_val")
            ).filter(col(col_name).isNotNull() & col("cast_val").isNull())

            # StringType
            if test.limit(1).count() == 0:
                found_type = t
                break
        
        best_types[col_name] = found_type

    # Final transformation
    for col_name, dtype in best_types.items():
        df = df.withColumn(col_name, col(col_name).cast(dtype))
    
    return df

In [0]:
def write_silver(catalog, domain):
    """
    Orchestrates the Bronze-to-Silver movement with flattening and typing.
    """
    # Fetch tables from Information Schema
    tables_df = spark.read.table(f"{catalog}.information_schema.tables")\
                    .where(col("table_schema") == domain)\
                    .where(col("table_name").startswith("bronze"))\
                    .select(concat_ws(".", col("table_catalog"), col("table_schema"), col("table_name")).alias("full_table_path"))

    table_paths = [row.full_table_path for row in tables_df.collect()]

    for table in table_paths:
        write_path = table.replace("bronze", "silver")

        try:
            # Check if table exists and get max timestamp
            last_ts = spark.table(write_path).select(max("_ingest_timestamp")).collect()[0][0]
            silver_exists = True
        except AnalysisException:
            last_ts = None
            silver_exists = False

        # Load Data
        if last_ts:
            raw_df = spark.read.table(table).where(col("_ingest_timestamp") > last_ts)
        else:
            raw_df = spark.read.table(table)

        if raw_df.limit(1).count() == 0:
            print(f"No new data for {table}, table is skipped.")
            continue
        
        print(f"\nProcessing {table}...")

        # Flatten
        silver_df = recursive_explode(raw_df)
        print(f"Flattened to {len(silver_df.columns)} columns")

        if silver_exists:
            # Get existing schema columns
            existing_columns = spark.catalog.listColumns(write_path)
            existing_col_names = [c.name for c in existing_columns]
            
            for c in existing_columns:
                if c.name in silver_df.columns:
                    silver_df = silver_df.withColumn(c.name, col(c.name).cast(c.dataType))

            # Account for new columns from exploding
            new_cols = [c for c in silver_df.columns if c not in existing_col_names]
            if new_cols:
                # Temporary DF with just new columns to infer their types
                temp_inferred = infer_datatypes(silver_df.select(*new_cols))
                for nc in new_cols:
                    new_type = temp_inferred.schema[nc].dataType
                    silver_df = silver_df.withColumn(nc, col(nc).cast(new_type))
        else:
            # First run
            silver_df = infer_datatypes(silver_df)

        silver_df = silver_df.withColumn("process_dts", current_timestamp())

        # 4. Write
        silver_df.write \
            .format("delta") \
            .mode("append") \
            .option("mergeSchema", "true") \
            .saveAsTable(write_path)

        print(f"Successfully processed Silver table: {write_path}")

In [0]:
write_silver(dbutils.widgets.get("CATALOG"), dbutils.widgets.get("DOMAIN"))